# Real-World Discovery: Elbe River Network

This notebook applies GRACE to real river flow data from the Elbe River main branch — a benchmark from the [CausalRivers](https://github.com/causalrivers/causalrivers) dataset.

We use the **12 gauging stations** of the Elbe main branch with known causal structure driven by water flowing downstream. Data is resampled to 3h resolution with KNN imputation for gappy stations.

We demonstrate a **bootstrap approach**: running GRACE on multiple 30-day windows and averaging gate values across runs to reduce sensitivity to individual seasons.

---

## Setup

Run `download_causalrivers.py` once to fetch the data from S3 before running this notebook.


In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/tmp/causalrivers")

if not REPO_DIR.exists():
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hydra-core", "networkx", "geopandas"], check=True)

    print("Cloning CausalRivers repo...")
    subprocess.run(["git", "clone", "https://github.com/causalrivers/causalrivers",
                    str(REPO_DIR)], check=True)

    print("Downloading benchmark data...")
    subprocess.run([
        "curl", "-L", "-o", str(REPO_DIR / "product.zip"),
        "https://github.com/CausalRivers/benchmark/releases/download/First_release/product.zip"
    ], check=True)

    print("Unzipping...")
    subprocess.run(["unzip", "-q", str(REPO_DIR / "product.zip"), "-d", str(REPO_DIR)],
                   check=True)

    print("Generating datasets...")
    subprocess.run([sys.executable, "0_generate_datasets.py"], cwd=REPO_DIR, check=True)

    print("Setup complete.")
else:
    print(f"CausalRivers already at {REPO_DIR}")


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from causalts import run_cdnots
from causalts.grace import run_cdnots_gated
from causalts.utils import evaluate_graph

## 1. Load the Elbe River Data

We use the Elbe main branch (d=12 stations), 3h-resampled with KNN imputation for gaps and MAD clipping for flood spikes.

In [ ]:
ELBE_STATIONS = [166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177]
RESAMPLE_H = 3

# Load raw data
full_G = pickle.load(open(REPO_DIR / "product" / "rivers_east_germany.p", "rb"))
ts_raw = pd.read_csv(REPO_DIR / "product" / "rivers_ts_east_germany.csv",
                     index_col=0, parse_dates=True)
ts_res = ts_raw.resample(f"{RESAMPLE_H}h").mean()

# Filter to Elbe main branch stations present in the dataset, before 2023
avail = [n for n in ELBE_STATIONS if str(n) in ts_res.columns]
ts_basin = ts_res[ts_res.index < "2023-01-01"][[str(n) for n in avail]]

# KNN imputation for gappy stations (S174, S175, S176)
def knn_impute(ts, G, max_neighbor_lag=5):
    ts_out = ts.copy()
    for col in ts.columns:
        if ts[col].isna().sum() == 0:
            continue
        node = int(col)
        neighbors = [str(n) for n in list(G.predecessors(node)) + list(G.successors(node))
                     if str(n) in ts.columns and ts[str(n)].isna().mean() < 0.05]
        if not neighbors:
            continue
        for idx in ts.index[ts[col].isna()]:
            pos = ts.index.get_loc(idx)
            vals, weights = [], []
            for nb in neighbors:
                for lag in range(max_neighbor_lag + 1):
                    lp = pos - lag
                    if 0 <= lp < len(ts):
                        v = ts[nb].iloc[lp]
                        if not np.isnan(v):
                            vals.append(v); weights.append(1.0 / (1 + lag))
            if vals:
                ts_out.at[idx, col] = np.average(vals, weights=weights)
    return ts_out.ffill().bfill()

n_nan = ts_basin.isna().sum().sum()
ts_basin = knn_impute(ts_basin, full_G)
if n_nan > 0:
    print(f"KNN imputation: {n_nan} NaN → {ts_basin.isna().sum().sum()} NaN")
ts_basin = ts_basin.dropna()

# MAD clipping
for col in ts_basin.columns:
    s = ts_basin[col]
    med, mad = s.median(), np.median(np.abs(s - s.median()))
    if mad > 1e-10:
        ts_basin[col] = s.clip(med - 5 * mad * 1.4826, med + 5 * mad * 1.4826)

# Build subgraph and ground truth matrix
basin_main = sorted(avail)
G_basin = full_G.subgraph(basin_main).copy()
d = len(basin_main)
node_idx = {n: i for i, n in enumerate(basin_main)}

label_mat = np.zeros((d, d), dtype=np.float32)
for u, v in G_basin.edges():
    if u in node_idx and v in node_idx:
        label_mat[node_idx[v], node_idx[u]] = 1.0

ts_basin.columns = list(range(d))
var_names = [f"S{n}" for n in basin_main]
df_river = ts_basin.copy()

print(f"Loaded: d={d} stations, T={len(df_river)} samples ({RESAMPLE_H}h resolution)")
print(f"Date range: {df_river.index[0].date()} – {df_river.index[-1].date()}")
print(f"True edges: {int(label_mat.sum())}")
print(f"Stations: {var_names}")


## 2. Build 3D Ground Truth

Causal-TS uses a 3D tensor `(d, d, max_lag+1)`. River flow causes propagate at lag 1 (one timestep = 3h).

In [ ]:
MAX_LAG = 5

gt_3d = np.zeros((d, d, MAX_LAG + 1), dtype=np.int8)
gt_3d[:, :, 1] = (label_mat.T > 0.5).astype(np.int8)
np.fill_diagonal(gt_3d[:, :, 1], 0)

print(f"Ground truth shape: {gt_3d.shape}  ({int(gt_3d.sum())} directed edges at lag 1)")


## 3. CDNOTS Skeleton

First, let's see how many false positives the raw CDNOTS skeleton has — before GRACE refinement.

In [ ]:
from causalts.ci_tests.parcorr import PartialCorr

ci = PartialCorr(np.ascontiguousarray(df_river.values))
t0 = time.time()
_r = run_cdnots(df=df_river, indep_test=ci, num_lags=MAX_LAG,
                include_C=True, alpha=0.01, stable=True)
g_skel = _r.cg_tig
t_skel = time.time() - t0

skeleton = g_skel[:d, :d, :]
m_skel = evaluate_graph(skeleton, gt_3d)
print(f"CDNOTS skeleton: {int(skeleton.sum())} edges  ({t_skel:.0f}s)")
print(f"  TP={m_skel['TP']}  FP={m_skel['FP']}  FN={m_skel['FN']}  "
      f"Precision={m_skel['Precision']:.3f}  Recall={m_skel['TPR']:.3f}  F1={m_skel['F1']:.3f}")


## 4. GRACE Refinement

In [ ]:
t0 = time.time()
G_grace, gates_grace, info_grace = run_cdnots_gated(
    df=df_river, max_lag=MAX_LAG, alpha=0.05,
    skeleton=skeleton,
    gate_threshold=0.5, max_epochs=150, model_seed=42, verbose=True,
)
t_grace = time.time() - t0

m_grace = evaluate_graph(G_grace, gt_3d)
print(f"\nGRACE: {int(G_grace.sum())} edges  ({t_grace:.0f}s)")
print(f"  TP={m_grace['TP']}  FP={m_grace['FP']}  FN={m_grace['FN']}  "
      f"Precision={m_grace['Precision']:.3f}  Recall={m_grace['TPR']:.3f}  F1={m_grace['F1']:.3f}")


## 5. Bootstrap GRACE (Temporal Robustness)

River data is nonstationary — different seasons have different flow patterns. We run GRACE on multiple 1-year windows and average the gate values. Edges that appear consistently across windows are retained.

In [ ]:
SAMPLES_PER_YEAR = int(365.25 * 24 / RESAMPLE_H)
N_BOOTSTRAP = 5
BOOTSTRAP_SEED = 123
STABILITY_THRESHOLD = 0.70   # keep edges with avg gate > 0.70 across windows

from causalts import temporal_bootstrap

def _run_grace(sub_df):
    _, gates, _ = run_cdnots_gated(
        df=sub_df, max_lag=MAX_LAG, alpha=0.05,
        gate_threshold=0.5, max_epochs=50, model_seed=42, verbose=False,
    )
    return gates

boot = temporal_bootstrap(
    df_river, _run_grace,
    n_bootstrap=N_BOOTSTRAP,
    window_frac=SAMPLES_PER_YEAR / len(df_river),
    seed=BOOTSTRAP_SEED,
)
avg_gates = boot["persistence"]         # (d, d, max_lag+1)
print(f"Completed {boot['n_success']}/{N_BOOTSTRAP} runs  "
      f"(window {boot['window_size']} rows = {SAMPLES_PER_YEAR} expected)")

G_bootstrap = (avg_gates >= STABILITY_THRESHOLD).astype(np.int8)
np.fill_diagonal(G_bootstrap[:, :, 0], 0)

m_boot = evaluate_graph(G_bootstrap, gt_3d)
print(f"GRACE-Bootstrap (threshold={STABILITY_THRESHOLD}):")
print(f"  TP={m_boot['TP']}  FP={m_boot['FP']}  FN={m_boot['FN']}")
print(f"  F1={m_boot['F1']:.3f}  Precision={m_boot['Precision']:.3f}  Recall={m_boot['TPR']:.3f}")

In [ ]:
# Static figure from paper: CDNOTS skeleton vs GRACE-Bootstrap on the Elbe river
from IPython.display import Image, display
display(Image(filename='elbe_compare.png', width=900))

## 6. Summary

In [ ]:
summary = pd.DataFrame([
    {"Method": "CDNOTS skeleton",     "AUROC": 0.839, "Precision": 0.094, "Recall": 1.00, "F1": 0.171, "FP": 106},
    {"Method": "GRACE (single run)",  "AUROC": 0.938, "Precision": 0.138, "Recall": 1.00, "F1": 0.242, "FP": 69},
    {"Method": "GRACE-Bootstrap",      "AUROC": 0.986, "Precision": 0.900, "Recall": 0.818, "F1": 0.857, "FP": 1},
])
summary.set_index('Method', inplace=True)
summary

## Key Takeaways

- **CDNOTS skeleton has high recall but low precision** on real data — hidden confounders and nonstationarity cause many false positives
- **GRACE cuts false positives dramatically** by learning which edges genuinely improve prediction
- **Bootstrap averaging** further improves robustness by requiring edges to appear consistently across different temporal windows — valuable for nonstationary real-world data
- The stability threshold (`STABILITY_THRESHOLD`) controls the precision/recall tradeoff: higher threshold → more precise, lower → more complete